# 🧪 Model Experiments: Scratch Gaussian Naive Bayes vs. Scikit-Learn
**Author:** Person B (@aakritishakya9)  
**Environment:** Google Colab  
**Objective:** Train our NumPy `ScratchGaussianNB` model on DataCo, benchmark against `sklearn.naive_bayes.GaussianNB`, evaluate High-Risk Recall, and test decision threshold tuning for procurement risk.

## 1. Cloud Setup & Data Loading
Download and load the DataCo dataset directly inside Colab.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, confusion_matrix, recall_score, precision_score, f1_score, accuracy_score
from sklearn.naive_bayes import GaussianNB as SklearnGaussianNB
import time

# Ensure dataset is downloaded
!mkdir -p ./data_raw
if not os.path.exists('./data_raw/DataCoSupplyChainDataset.csv'):
    print("⏳ Downloading DataCo dataset...")
    !wget -q --show-progress -O ./data_raw/DataCoSupplyChainDataset.csv "https://huggingface.co/idoyaaran/dataco-supply-chain-model/resolve/main/DataCoSupplyChainDataset.csv"

df = pd.read_csv('./data_raw/DataCoSupplyChainDataset.csv', encoding='latin-1')
print(f"✅ Data loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")

## 2. Feature Selection & Preprocessing Pipeline (Leakage-Safe)
We select order-time features only and strip any outcome leakage variables (`Days for shipping (real)`, `Delivery Status`).

In [ ]:
TARGET_COL = 'Late_delivery_risk'

NUMERIC_FEATURES = [
    'Days for shipment (scheduled)',
    'Benefit per order',
    'Sales per customer',
    'Order Item Discount',
    'Order Item Discount Rate',
    'Order Item Product Price',
    'Order Item Profit Ratio',
    'Order Item Quantity',
    'Sales',
    'Product Price'
]

CATEGORICAL_FEATURES = [
    'Shipping Mode',
    'Customer Segment',
    'Market'
]

# Clean and filter rows with valid target
df_clean = df.dropna(subset=[TARGET_COL]).copy()

X = df_clean[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = df_clean[TARGET_COL].values.astype(int)

# Stratified 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Transform features: Standardize numerics, One-Hot encode categoricals
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), NUMERIC_FEATURES),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL_FEATURES)
    ]
)

X_train_trans = preprocessor.fit_transform(X_train)
X_test_trans = preprocessor.transform(X_test)

print(f"✅ Training set shape:   {X_train_trans.shape}")
print(f"✅ Test set shape:       {X_test_trans.shape}")

## 3. Person B: Your Scratch Gaussian Naive Bayes (NumPy)
Here is the exact mathematical implementation of Gaussian Naive Bayes from your repository (`src/models/naive_bayes.py`).

In [ ]:
class ScratchGaussianNB:
    def __init__(self, var_smoothing=1e-9):
        self.var_smoothing = var_smoothing
        self.classes_ = None
        self.class_priors_ = None
        self.means_ = None
        self.variances_ = None

    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y)
        n_samples, n_features = X.shape
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)

        self.means_ = np.zeros((n_classes, n_features))
        self.variances_ = np.zeros((n_classes, n_features))
        self.class_priors_ = np.zeros(n_classes)

        # Numerical stability epsilon
        global_max_var = np.var(X, axis=0).max()
        epsilon = self.var_smoothing * global_max_var

        for idx, c in enumerate(self.classes_):
            X_c = X[y == c]
            self.class_priors_[idx] = X_c.shape[0] / n_samples
            self.means_[idx, :] = np.mean(X_c, axis=0)
            self.variances_[idx, :] = np.var(X_c, axis=0) + epsilon
        return self

    def _calculate_log_likelihood(self, X):
        n_samples = X.shape[0]
        n_classes = len(self.classes_)
        log_posteriors = np.zeros((n_samples, n_classes))

        for idx in range(n_classes):
            mean = self.means_[idx]
            var = self.variances_[idx]
            prior = self.class_priors_[idx]

            log_prior = np.log(prior)
            log_likelihood = -0.5 * np.sum(np.log(2.0 * np.pi * var)) - 0.5 * np.sum(
                ((X - mean) ** 2) / var, axis=1
            )
            log_posteriors[:, idx] = log_prior + log_likelihood
        return log_posteriors

    def predict_proba(self, X):
        X = np.asarray(X, dtype=np.float64)
        log_post = self._calculate_log_likelihood(X)
        max_log = np.max(log_post, axis=1, keepdims=True)
        exp_post = np.exp(log_post - max_log)
        return exp_post / np.sum(exp_post, axis=1, keepdims=True)

    def predict(self, X, threshold=0.5):
        probs = self.predict_proba(X)
        return (probs[:, 1] >= threshold).astype(int)

print("✅ ScratchGaussianNB class defined!")

## 4. Train Both Models: Scratch vs. Scikit-Learn

In [ ]:
# 1. Train Scratch Gaussian NB
t0 = time.time()
scratch_nb = ScratchGaussianNB(var_smoothing=1e-9)
scratch_nb.fit(X_train_trans, y_train)
scratch_fit_time = time.time() - t0

t0 = time.time()
y_pred_scratch = scratch_nb.predict(X_test_trans)
y_prob_scratch = scratch_nb.predict_proba(X_test_trans)[:, 1]
scratch_pred_time = time.time() - t0

# 2. Train Scikit-Learn Gaussian NB
t0 = time.time()
sklearn_nb = SklearnGaussianNB(var_smoothing=1e-9)
sklearn_nb.fit(X_train_trans, y_train)
sklearn_fit_time = time.time() - t0

t0 = time.time()
y_pred_sklearn = sklearn_nb.predict(X_test_trans)
y_prob_sklearn = sklearn_nb.predict_proba(X_test_trans)[:, 1]
sklearn_pred_time = time.time() - t0

print("✅ Both models trained successfully!")

## 5. Verification & Parity Check
Do our scratch predictions match `scikit-learn`?

In [ ]:
# Check prediction agreement rate between scratch and sklearn
agreement = np.mean(y_pred_scratch == y_pred_sklearn) * 100
max_prob_diff = np.max(np.abs(y_prob_scratch - y_prob_sklearn))

print(f"🎯 Prediction Agreement Rate: {agreement:.2f}%")
print(f"🔍 Max Absolute Probability Difference: {max_prob_diff:.6f}")

if agreement > 99.0:
    print("🎉 OUTSTANDING: Your NumPy Scratch model matches scikit-learn parity!")
else:
    print("Note: Slight difference due to internal variance smoothing.")

## 6. Evaluation Metrics: Focus on High-Risk Recall
In procurement, missing a late supplier (False Negative) is costly. Let's inspect Accuracy, Precision, Recall, and F1.

In [ ]:
metrics_data = {
    'Metric': ['Accuracy', 'High-Risk Recall (Late=1)', 'Precision (Late=1)', 'F1-Score (Late=1)', 'Training Time (s)'],
    'Scratch Gaussian NB': [
        accuracy_score(y_test, y_pred_scratch),
        recall_score(y_test, y_pred_scratch, pos_label=1),
        precision_score(y_test, y_pred_scratch, pos_label=1),
        f1_score(y_test, y_pred_scratch, pos_label=1),
        scratch_fit_time
    ],
    'Scikit-Learn Gaussian NB': [
        accuracy_score(y_test, y_pred_sklearn),
        recall_score(y_test, y_pred_sklearn, pos_label=1),
        precision_score(y_test, y_pred_sklearn, pos_label=1),
        f1_score(y_test, y_pred_sklearn, pos_label=1),
        sklearn_fit_time
    ]
}

df_metrics = pd.DataFrame(metrics_data)
print(df_metrics.to_string(index=False))

# Confusion Matrix comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
cm_scratch = confusion_matrix(y_test, y_pred_scratch)
cm_sklearn = confusion_matrix(y_test, y_pred_sklearn)

sns.heatmap(cm_scratch, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['On-Time (0)', 'Late (1)'], yticklabels=['On-Time (0)', 'Late (1)'])
axes[0].set_title("Scratch Gaussian NB Confusion Matrix")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

sns.heatmap(cm_sklearn, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=['On-Time (0)', 'Late (1)'], yticklabels=['On-Time (0)', 'Late (1)'])
axes[1].set_title("Scikit-Learn Gaussian NB Confusion Matrix")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")

plt.tight_layout()
plt.show()

## 7. Business Optimization: Threshold Tuning for High-Risk Recall
Can we catch even more high-risk suppliers by lowering the risk threshold from 0.50 to 0.35?

In [ ]:
thresholds = [0.30, 0.40, 0.50, 0.60, 0.70]
recalls, precisions = [], []

for t in thresholds:
    y_t = scratch_nb.predict(X_test_trans, threshold=t)
    recalls.append(recall_score(y_test, y_t, pos_label=1))
    precisions.append(precision_score(y_test, y_t, pos_label=1))

df_thresh = pd.DataFrame({
    'Decision Threshold': thresholds,
    'High-Risk Recall (Late=1)': [f"{r*100:.1f}%" for r in recalls],
    'Precision (Late=1)': [f"{p*100:.1f}%" for p in precisions]
})
print("📊 Risk Threshold Tuning Analysis:")
print(df_thresh.to_string(index=False))

plt.figure(figsize=(8, 5))
plt.plot(thresholds, recalls, marker='o', label='High-Risk Recall (Sensitivity)', color='red', lw=2)
plt.plot(thresholds, precisions, marker='s', label='Precision', color='blue', lw=2)
plt.axvline(0.50, color='gray', linestyle='--', label='Default Threshold (0.50)')
plt.title("Recall vs Precision Trade-off by Decision Threshold")
plt.xlabel("Risk Threshold")
plt.ylabel("Score")
plt.legend()
plt.grid(True)
plt.show()